# Pipeline de recertification KYC — Cartes d'identité & Justificatifs de domicile

## Objectif
Extraire automatiquement les informations des documents **JUSTIFICATIF IDENTITE.PDF** et
**JUSTIFICATIF DOMICILE.PDF** contenus dans les dossiers clients (nommés par leur ID) d'une
archive ZIP, puis **comparer** ces informations avec le référentiel `tiers.csv` pour détecter
les **incohérences**.

## Contraintes du cas d'usage
- Scans de mauvaise qualité, parfois des **photocopies de photocopies**.
- **Rotation variable** des documents (0°, 90°, 180°, 270°, voire légèrement de travers / *skew*).
- Documents de **pays différents**, formats hétérogènes.
- Documents **émis en Algérie** : bilingues **arabe/français**, structure assez stable →
  on peut utiliser un **schéma JSON strict par type de document** (carte d'identité nationale,
  justificatif de domicile algérien).
- Documents **émis hors Algérie** : formats très variables → **extraction libre (schema-less)**,
  on laisse le modèle renvoyer les champs qu'il détecte sous forme de JSON clé/valeur.

## Modèles disponibles (ModelHub, chemins locaux)
| Modèle | Rôle proposé |
|---|---|
| `PaddleOCR-VL` | OCR + détection de mise en page + **détection d'orientation** (très bon pour texte multi-script, y compris arabe). Utilisé en **prétraitement** pour redresser l'image et obtenir un texte brut de référence. |
| `Qwen2.5-VL-7B-Instruct` | Modèle principal d'**extraction structurée** (compréhension de documents, bon support multilingue arabe/français, bonne fidélité au JSON demandé). |
| `InternVL3-8B` | Modèle de **repli / vérification croisée** : si Qwen échoue à produire un JSON valide, ou pour un second passage de contrôle sur les champs les plus sensibles (numéro de pièce, date de naissance). |

## Stratégie technique retenue (résumé)
1. **Dézipper** l'archive et repérer, pour chaque dossier client (nommé par l'ID), les deux PDF
   cibles (recherche *fuzzy* du nom de fichier — accents, casse, espaces).
2. **PDF → image** haute résolution (300 DPI) par page utile.
3. **Prétraitement image** :
   - Détection/correction de rotation (OSD Tesseract si dispo *et* vérification/fallback via
     PaddleOCR-VL — utile car les scans sont bruités et l'OSD classique échoue parfois).
   - Redressement (*deskew*) léger via OpenCV.
   - Amélioration de contraste / netteté pour les photocopies très pâles.
4. **Détection du pays d'émission** (heuristique mots-clés bilingues + confirmation modèle).
5. **Extraction** :
   - Algérie → prompt **avec schéma JSON strict** par type de document (bilingue AR/FR).
   - Hors Algérie → prompt **libre**, JSON dynamique.
6. **Parsing robuste** du JSON renvoyé (le modèle peut ajouter du texte autour du JSON).
7. **Agrégation** en DataFrame : un enregistrement par client × type de document.
8. **Rapprochement avec `tiers.csv`** : normalisation des chaînes, matching *fuzzy* (nom/prénom),
   comparaison des dates, des adresses, du numéro de pièce → **rapport d'incohérences**.

> ⚠️ Ce notebook est écrit pour être exécuté dans l'environnement Domino (accès aux chemins
> `/domino/edv/modelhub/...`, GPU disponible). Les chemins ZIP / CSV sont à adapter dans la
> cellule de configuration.


In [ ]:
# (Optionnel) Installation des dépendances si non présentes dans l'environnement Domino
# %pip install -q pymupdf opencv-python-headless pillow pytesseract rapidfuzz unidecode \
#     pandas openpyxl transformers accelerate qwen-vl-utils torch
# NB: PaddleOCR-VL / Qwen2.5-VL / InternVL3 sont chargés directement depuis le ModelHub local,
# pas besoin de les télécharger.


In [ ]:
import os
import re
import io
import json
import zipfile
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
from transformers import AutoProcessor, AutoModelForCausalLM

try:
    from rapidfuzz import fuzz
except ImportError:
    fuzz = None

try:
    import pytesseract
    HAS_TESSERACT = True
except ImportError:
    HAS_TESSERACT = False

print("Torch CUDA disponible :", torch.cuda.is_available())


## 1. Configuration

In [ ]:
# --- Chemins des modèles (ModelHub) ---
QWEN_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main"
PADDLEOCR_VL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main"
INTERNVL3_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/internVL3-8B/main"

# --- Chemins des données ---
ZIP_PATH = "/mnt/data/dossiers_clients.zip"          # archive ZIP des dossiers clients
WORKDIR = Path("/tmp/kyc_extraction")                # dossier de travail (extraction du zip)
TIERS_CSV_PATH = "/mnt/data/tiers.csv"                # référentiel tiers
OUTPUT_DIR = Path("/mnt/data/output_kyc")             # rapports de sortie

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Noms cibles à rechercher dans chaque dossier client (recherche fuzzy, insensible
# à la casse / accents / espaces superflus)
TARGET_DOCS = {
    "identite": "JUSTIFICATIF IDENTITE",
    "domicile": "JUSTIFICATIF DOMICILE",
}

DPI = 300              # résolution de rendu PDF -> image
FUZZY_FILENAME_THRESHOLD = 80   # seuil de similarité (0-100) pour matcher un nom de fichier


## 2. Dézippage et repérage des 2 PDF cibles par client

Chaque dossier client (nommé par son ID) contient plusieurs PDF. On ne garde que les deux
fichiers qui correspondent (par similarité floue de nom) à `JUSTIFICATIF IDENTITE` et
`JUSTIFICATIF DOMICILE`, pour tolérer variations d'accents, de casse, d'espaces ou de légères
fautes de frappe dans les noms de fichiers réels.

In [ ]:
def normalize_text(s: str) -> str:
    \"\"\"Supprime accents, met en majuscules, compresse les espaces.\"\"\"
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip().upper()
    return s


def fuzzy_score(a: str, b: str) -> int:
    a, b = normalize_text(a), normalize_text(b)
    if fuzz is not None:
        return fuzz.partial_ratio(a, b)
    # fallback naïf si rapidfuzz absent
    return 100 if b in a else 0


def extract_zip(zip_path: str, workdir: Path) -> Path:
    extract_to = workdir / "extracted"
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)
    return extract_to


def list_client_folders(extract_root: Path):
    \"\"\"Retourne la liste des dossiers clients (1 niveau, ou recherche récursive si zip imbriqué).\"\"\"
    folders = [p for p in extract_root.iterdir() if p.is_dir()]
    if not folders:
        # cas où le zip contient directement les dossiers à un niveau plus profond
        folders = [p for p in extract_root.rglob("*") if p.is_dir()]
    return folders


def find_target_pdfs(client_folder: Path) -> dict:
    \"\"\"Cherche dans le dossier client les 2 PDF cibles par similarité de nom.
    Retourne {'identite': Path|None, 'domicile': Path|None}\"\"\"
    result = {"identite": None, "domicile": None}
    pdf_files = list(client_folder.rglob("*.pdf")) + list(client_folder.rglob("*.PDF"))
    for key, target_name in TARGET_DOCS.items():
        best_path, best_score = None, 0
        for pdf in pdf_files:
            score = fuzzy_score(pdf.stem, target_name)
            if score > best_score:
                best_score, best_path = score, pdf
        if best_score >= FUZZY_FILENAME_THRESHOLD:
            result[key] = best_path
    return result


def build_client_index(zip_path: str, workdir: Path) -> pd.DataFrame:
    extract_root = extract_zip(zip_path, workdir)
    folders = list_client_folders(extract_root)
    rows = []
    for folder in folders:
        client_id = folder.name
        targets = find_target_pdfs(folder)
        rows.append({
            "client_id": client_id,
            "path_identite": str(targets["identite"]) if targets["identite"] else None,
            "path_domicile": str(targets["domicile"]) if targets["domicile"] else None,
        })
    df = pd.DataFrame(rows)
    missing = df[df["path_identite"].isna() | df["path_domicile"].isna()]
    if len(missing):
        print(f"⚠️  {len(missing)} client(s) avec au moins un document cible manquant :")
        print(missing[["client_id"]].to_string(index=False))
    return df

# client_index = build_client_index(ZIP_PATH, WORKDIR)
# client_index.head()


## 3. Conversion PDF → image(s) haute résolution

In [ ]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path: str, dpi: int = DPI):
    \"\"\"Convertit chaque page d'un PDF en image PIL (RGB).\"\"\"
    images = []
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=mat, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            images.append(img)
    return images


def pick_main_page(images):
    \"\"\"Heuristique simple: garde la page avec le plus de contenu (variance de pixels)
    -> évite les pages blanches/quasi-vides parfois insérées par le scanner.\"\"\"
    if len(images) == 1:
        return images[0]
    scored = []
    for img in images:
        arr = np.array(img.convert("L"))
        scored.append((arr.std(), img))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


## 4. Prétraitement image : rotation, redressement, amélioration de lisibilité

Les scans/photocopies présentent des rotations arbitraires (0/90/180/270) et parfois un léger
biais (*skew*). On combine :
- **Tesseract OSD** (rapide) quand le texte est encore assez net,
- un **fallback par le modèle vision-langage** (on demande directement au VLM l'orientation du
  document) quand l'image est trop dégradée pour l'OSD classique — ce qui est fréquent sur des
  photocopies de photocopies.

In [ ]:
def deskew(image: Image.Image) -> Image.Image:
    \"\"\"Redressement fin (petit angle) via détection de contours + minAreaRect.\"\"\"
    arr = np.array(image.convert("L"))
    arr = cv2.GaussianBlur(arr, (5, 5), 0)
    _, thresh = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.5 or abs(angle) > 20:
        return image  # pas de correction si angle négligeable ou aberrant
    (h, w) = arr.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(np.array(image), M, (w, h),
                              flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rotated)


def enhance_for_ocr(image: Image.Image) -> Image.Image:
    \"\"\"Amélioration contraste/netteté pour photocopies pâles ou grisées.\"\"\"
    arr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    arr = clahe.apply(arr)
    arr = cv2.fastNlMeansDenoising(arr, h=10)
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB))


def detect_rotation_tesseract(image: Image.Image):
    if not HAS_TESSERACT:
        return None
    try:
        osd = pytesseract.image_to_osd(image)
        m = re.search(r"Rotate: (\d+)", osd)
        conf = re.search(r"Orientation confidence: ([\d.]+)", osd)
        if m and conf and float(conf.group(1)) >= 1.0:
            return int(m.group(1))
    except Exception:
        return None
    return None


def rotate_image(image: Image.Image, angle: int) -> Image.Image:
    if angle % 360 == 0:
        return image
    return image.rotate(-angle, expand=True)  # rotation horaire pour corriger


def detect_rotation_vlm(image: Image.Image, ask_vlm_fn) -> int:
    \"\"\"Fallback: demande au VLM l'angle de rotation nécessaire (0/90/180/270)
    pour remettre le document à l'endroit. ask_vlm_fn(image, prompt) -> str\"\"\"
    prompt = (
        "Regarde ce document scanné (carte d'identité ou justificatif de domicile). "
        "Indique uniquement l'angle de rotation horaire nécessaire pour que le texte soit "
        "parfaitement à l'endroit et lisible. Réponds strictement par un seul nombre parmi "
        "0, 90, 180, 270."
    )
    answer = ask_vlm_fn(image, prompt)
    match = re.search(r"\b(0|90|180|270)\b", answer)
    return int(match.group(1)) if match else 0


def auto_orient_and_clean(image: Image.Image, ask_vlm_fn=None) -> Image.Image:
    angle = detect_rotation_tesseract(image)
    if angle is None and ask_vlm_fn is not None:
        angle = detect_rotation_vlm(image, ask_vlm_fn)
    if angle:
        image = rotate_image(image, angle)
    image = deskew(image)
    image = enhance_for_ocr(image)
    return image


## 5. Chargement des modèles (Qwen2.5-VL principal, PaddleOCR-VL & InternVL3 en appui)

- **Qwen2.5-VL-7B-Instruct** : extraction structurée principale.
- **PaddleOCR-VL** : OCR brut / détection de mise en page, utile en amont pour valider
  l'orientation et fournir un texte de référence en cas de doute.
- **InternVL3-8B** : modèle de repli si Qwen échoue à produire un JSON exploitable, ou pour un
  second avis sur les champs critiques.

> Pour économiser la VRAM, les modèles peuvent être chargés **à la demande** plutôt que tous en
> mémoire simultanément (voir `load_model` avec un cache paresseux).

In [ ]:
_MODEL_CACHE = {}

def load_model(model_path: str):
    \"\"\"Charge (une seule fois) le modèle et son processor depuis un chemin local ModelHub.\"\"\"
    if model_path in _MODEL_CACHE:
        return _MODEL_CACHE[model_path]

    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

    if "Qwen2.5-VL" in model_path or "Qwen2_5" in model_path:
        from transformers import Qwen2_5_VLForConditionalGeneration
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
        )
    else:
        # InternVL3 / PaddleOCR-VL exposés via AutoModelForCausalLM (remote code du repo)
        model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
        )
    model.eval()
    _MODEL_CACHE[model_path] = (model, processor)
    return model, processor


def unload_model(model_path: str):
    \"\"\"Libère la VRAM d'un modèle si nécessaire (utile en environnement GPU limité).\"\"\"
    if model_path in _MODEL_CACHE:
        model, _ = _MODEL_CACHE.pop(model_path)
        del model
        torch.cuda.empty_cache()


## 6. Fonction générique d'appel à un modèle vision-langage

In [ ]:
def call_vlm(image: Image.Image, prompt: str, model_path: str = QWEN_PATH,
             max_new_tokens: int = 1024, temperature: float = 0.0) -> str:
    \"\"\"Envoie une image + un prompt texte à un VLM (Qwen2.5-VL / InternVL3) et renvoie la
    réponse texte brute.\"\"\"
    model, processor = load_model(model_path)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    try:
        # Qwen2.5-VL : utilitaire officiel de préparation des images (si installé)
        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
    except ImportError:
        image_inputs, video_inputs = [image], None

    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return output_text.strip()


## 7. Détection du pays d'émission (Algérie vs autre)

Heuristique rapide sur mots-clés bilingues typiques des documents algériens, combinée à une
confirmation par le modèle en cas de doute (score heuristique faible).

In [ ]:
DZ_KEYWORDS = [
    "REPUBLIQUE ALGERIENNE", "REPUBLIQUE ALGERIENNE DEMOCRATIQUE ET POPULAIRE",
    "الجمهورية الجزائرية", "وزارة الداخلية", "BLIDA", "ALGER", "ORAN", "CONSTANTINE",
    "WILAYA", "COMMUNE DE", "الجزائر", "بطاقة التعريف الوطنية", "CARTE NATIONALE D'IDENTITE",
]

def heuristic_country_score(raw_text: str) -> float:
    text_norm = normalize_text(raw_text)
    hits = sum(1 for kw in DZ_KEYWORDS if normalize_text(kw) in text_norm)
    return min(hits / 2.0, 1.0)  # score 0-1


def detect_country(image: Image.Image, raw_ocr_text: str = "") -> str:
    \"\"\"Retourne 'DZ' ou 'OTHER'. Utilise d'abord le texte OCR brut (rapide), puis
    interroge le VLM en cas d'ambiguïté.\"\"\"
    score = heuristic_country_score(raw_ocr_text)
    if score >= 0.5:
        return "DZ"
    if raw_ocr_text and score == 0.0:
        return "OTHER"

    prompt = (
        "Ce document est-il une pièce d'identité ou un justificatif de domicile émis en "
        "Algérie ? Réponds strictement par 'DZ' si oui, ou 'OTHER' si le document provient "
        "d'un autre pays."
    )
    answer = call_vlm(image, prompt)
    return "DZ" if "DZ" in answer.upper() else "OTHER"


## 8. Schémas d'extraction pour les documents émis en Algérie

Un schéma JSON strict par type de document, avec libellés bilingues pour guider le modèle sur
un contenu mixte arabe/français.

In [ ]:
SCHEMA_DZ_IDENTITE = {
    "type_document": "carte_identite_nationale | passeport | permis_conduire",
    "nom": "string (nom de famille, تلقب)",
    "prenom": "string (prénom, الاسم)",
    "date_naissance": "YYYY-MM-DD",
    "lieu_naissance": "string",
    "sexe": "M | F",
    "numero_piece": "string (numéro de la carte / passeport)",
    "date_delivrance": "YYYY-MM-DD",
    "date_expiration": "YYYY-MM-DD",
    "autorite_delivrance": "string (wilaya / commune / daira)",
    "adresse": "string (adresse figurant sur la pièce, si présente)",
}

SCHEMA_DZ_DOMICILE = {
    "type_document": "facture_sonelgaz | facture_ADE | attestation_communale | autre",
    "nom_titulaire": "string",
    "prenom_titulaire": "string",
    "adresse_complete": "string",
    "commune": "string",
    "wilaya": "string",
    "code_postal": "string ou null",
    "date_document": "YYYY-MM-DD",
    "numero_reference": "string ou null (n° facture / n° acte)",
}

def build_schema_prompt(schema: dict, doc_label: str) -> str:
    schema_json = json.dumps(schema, ensure_ascii=False, indent=2)
    return f\"\"\"Tu es un expert en extraction de données KYC. Le document ci-joint est un
{doc_label} émis en Algérie. Le texte peut être en arabe, en français, ou bilingue.

Extrait les informations selon EXACTEMENT le schéma JSON suivant (mêmes clés). Si une
information est absente ou illisible, mets la valeur null. Ne complète jamais un champ par
une supposition. Traduis/normalise les dates au format YYYY-MM-DD.

Schéma attendu :
{schema_json}

Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ni après, sans balises
markdown.\"\"\"


## 9. Prompt d'extraction générique pour les documents hors Algérie

Pas de schéma imposé : le modèle renvoie librement les champs qu'il identifie, sous forme de
JSON clé/valeur, en s'appuyant sur sa connaissance générale des pièces d'identité et
justificatifs de domicile internationaux.

In [ ]:
def build_generic_prompt(doc_label: str) -> str:
    return f\"\"\"Tu es un expert en extraction de données KYC. Le document ci-joint est un
{doc_label} émis dans un pays autre que l'Algérie (format inconnu à l'avance).

Identifie le pays d'émission et le type précis de document, puis extrait toutes les
informations pertinentes que tu peux lire avec certitude (nom, prénom, date de naissance,
numéro de pièce, dates de délivrance/expiration, adresse complète, autorité émettrice, etc.).

Réponds UNIQUEMENT avec un objet JSON de la forme :
{{
  "pays_detecte": "...",
  "type_document": "...",
  "champs": {{ "nom_du_champ": "valeur", ... }}
}}

N'invente aucune valeur : si un champ est illisible ou absent, ne l'inclus pas. Normalise les
dates au format YYYY-MM-DD quand c'est possible. Pas de texte hors du JSON, pas de markdown.\"\"\"


## 10. Parsing robuste du JSON renvoyé par le modèle

In [ ]:
def parse_json_safe(raw_output: str):
    \"\"\"Extrait et parse le premier objet JSON valide trouvé dans la réponse du modèle.\"\"\"
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```json\s*|\s*```$", "", cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r"^```\s*|\s*```$", "", cleaned, flags=re.MULTILINE)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


## 11. Orchestration de l'extraction pour un document

Pipeline complet : préparation image → détection pays → choix du prompt (schéma DZ ou libre) →
appel au modèle principal (Qwen2.5-VL) → parsing → **repli sur InternVL3-8B** si le JSON est
invalide ou vide après 2 tentatives.

In [ ]:
def extract_document(pdf_path: str, doc_type: str) -> dict:
    \"\"\"doc_type: 'identite' ou 'domicile'\"\"\"
    result = {
        "doc_type": doc_type,
        "pdf_path": pdf_path,
        "pays_detecte": None,
        "modele_utilise": None,
        "extraction_ok": False,
        "data": None,
        "erreur": None,
    }
    try:
        images = pdf_to_images(pdf_path, dpi=DPI)
        raw_image = pick_main_page(images)
    except Exception as e:
        result["erreur"] = f"Erreur lecture PDF: {e}"
        return result

    ask_vlm_fn = lambda img, prompt: call_vlm(img, prompt, model_path=QWEN_PATH)
    clean_image = auto_orient_and_clean(raw_image, ask_vlm_fn=ask_vlm_fn)

    country = detect_country(clean_image)
    result["pays_detecte"] = country

    doc_label = "carte d'identité / pièce d'identité" if doc_type == "identite" else \
                "justificatif de domicile"

    if country == "DZ":
        schema = SCHEMA_DZ_IDENTITE if doc_type == "identite" else SCHEMA_DZ_DOMICILE
        prompt = build_schema_prompt(schema, doc_label)
    else:
        prompt = build_generic_prompt(doc_label)

    for attempt, model_path in enumerate([QWEN_PATH, INTERNVL3_PATH]):
        raw_output = call_vlm(clean_image, prompt, model_path=model_path)
        parsed = parse_json_safe(raw_output)
        if parsed:
            result["data"] = parsed
            result["extraction_ok"] = True
            result["modele_utilise"] = model_path
            break
        result["erreur"] = f"JSON invalide (tentative {attempt+1}) : {raw_output[:200]}"

    return result


## 12. Boucle principale sur tous les clients de l'archive

In [ ]:
def run_pipeline(zip_path: str, workdir: Path, checkpoint_every: int = 20) -> pd.DataFrame:
    client_index = build_client_index(zip_path, workdir)
    records = []
    checkpoint_path = OUTPUT_DIR / "extraction_checkpoint.jsonl"
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    for i, row in client_index.iterrows():
        client_id = row["client_id"]
        entry = {"client_id": client_id}

        for doc_type, path_col in [("identite", "path_identite"), ("domicile", "path_domicile")]:
            pdf_path = row[path_col]
            if not pdf_path:
                entry[f"{doc_type}_result"] = {"extraction_ok": False, "erreur": "Fichier absent"}
                continue
            res = extract_document(pdf_path, doc_type)
            entry[f"{doc_type}_result"] = res

        records.append(entry)

        # checkpoint incrémental (robustesse en cas d'interruption sur gros volumes)
        if (i + 1) % checkpoint_every == 0:
            with open(checkpoint_path, "a", encoding="utf-8") as f:
                for r in records[-checkpoint_every:]:
                    f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
            print(f"  ... {i+1}/{len(client_index)} clients traités")

    # flush final
    remaining = len(records) % checkpoint_every
    if remaining:
        with open(checkpoint_path, "a", encoding="utf-8") as f:
            for r in records[-remaining:]:
                f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

    return flatten_results(records)


def flatten_results(records) -> pd.DataFrame:
    rows = []
    for entry in records:
        row = {"client_id": entry["client_id"]}
        for doc_type in ["identite", "domicile"]:
            res = entry.get(f"{doc_type}_result", {})
            row[f"{doc_type}_ok"] = res.get("extraction_ok", False)
            row[f"{doc_type}_pays"] = res.get("pays_detecte")
            row[f"{doc_type}_modele"] = res.get("modele_utilise")
            data = res.get("data") or {}
            # aplatissement: schéma DZ (clés directes) vs schéma générique (sous "champs")
            if "champs" in data:
                flat = {**{k: v for k, v in data.items() if k != "champs"}, **data["champs"]}
            else:
                flat = data
            for k, v in flat.items():
                row[f"{doc_type}__{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)

# extraction_df = run_pipeline(ZIP_PATH, WORKDIR)
# extraction_df.to_csv(OUTPUT_DIR / "extraction_resultats.csv", index=False)
# extraction_df.head()


## 13. Rapprochement avec `tiers.csv` et détection des incohérences

On normalise les chaînes (accents, casse), on matche par `client_id`, puis on compare
champ à champ (nom, prénom, date de naissance, adresse, numéro de pièce) avec une tolérance
*fuzzy* pour absorber les petites variations d'OCR (ex: "MOHAMED" vs "MOHAMMED").

> ⚠️ Adapter les noms de colonnes ci-dessous à la structure réelle de `tiers.csv`.

In [ ]:
def load_tiers(tiers_csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(tiers_csv_path, dtype=str)
    df.columns = [c.strip().lower() for c in df.columns]
    return df


# Mapping colonnes extraction -> colonnes tiers.csv (À ADAPTER selon le référentiel réel)
FIELD_MAPPING = {
    "identite__nom": "nom",
    "identite__prenom": "prenom",
    "identite__date_naissance": "date_naissance",
    "identite__numero_piece": "numero_piece_identite",
    "domicile__adresse_complete": "adresse",
}


def norm_compare(a, b) -> float:
    \"\"\"Score de similarité 0-100 entre deux valeurs textuelles normalisées.\"\"\"
    a, b = normalize_text(str(a)), normalize_text(str(b))
    if not a or not b or a == "NONE" or b == "NONE":
        return 0.0
    if fuzz is not None:
        return fuzz.token_sort_ratio(a, b)
    return 100.0 if a == b else 0.0


def build_incoherence_report(extraction_df: pd.DataFrame, tiers_df: pd.DataFrame,
                              match_threshold: float = 85.0) -> pd.DataFrame:
    merged = extraction_df.merge(
        tiers_df, left_on="client_id", right_on="client_id", how="left",
        suffixes=("", "_tiers")
    )

    report_rows = []
    for _, row in merged.iterrows():
        incoherences = []
        for extr_col, tiers_col in FIELD_MAPPING.items():
            if extr_col not in row or tiers_col not in row:
                continue
            score = norm_compare(row.get(extr_col), row.get(tiers_col))
            if score < match_threshold:
                incoherences.append({
                    "champ": extr_col,
                    "valeur_extraite": row.get(extr_col),
                    "valeur_tiers": row.get(tiers_col),
                    "score_similarite": round(score, 1),
                })
        report_rows.append({
            "client_id": row["client_id"],
            "identite_extraction_ok": row.get("identite_ok"),
            "domicile_extraction_ok": row.get("domicile_ok"),
            "nb_incoherences": len(incoherences),
            "incoherences": incoherences,
        })

    report_df = pd.DataFrame(report_rows)
    return report_df.sort_values("nb_incoherences", ascending=False)


# tiers_df = load_tiers(TIERS_CSV_PATH)
# incoherence_report = build_incoherence_report(extraction_df, tiers_df)
# incoherence_report.head(20)


## 14. Export du rapport final (Excel avec mise en évidence des incohérences)

In [ ]:
def export_final_report(extraction_df: pd.DataFrame, incoherence_report: pd.DataFrame,
                          output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    xlsx_path = output_dir / f"rapport_kyc_{timestamp}.xlsx"

    # Détail des incohérences en format tabulaire (1 ligne par incohérence)
    detail_rows = []
    for _, r in incoherence_report.iterrows():
        for inc in r["incoherences"]:
            detail_rows.append({"client_id": r["client_id"], **inc})
    detail_df = pd.DataFrame(detail_rows)

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        extraction_df.to_excel(writer, sheet_name="extraction_brute", index=False)
        incoherence_report.drop(columns=["incoherences"]).to_excel(
            writer, sheet_name="synthese_incoherences", index=False
        )
        detail_df.to_excel(writer, sheet_name="detail_incoherences", index=False)

    print(f"Rapport exporté : {xlsx_path}")
    return xlsx_path

# export_final_report(extraction_df, incoherence_report, OUTPUT_DIR)


## 15. Exécution complète (à décommenter une fois les chemins vérifiés)

In [ ]:
# extraction_df = run_pipeline(ZIP_PATH, WORKDIR)
# extraction_df.to_csv(OUTPUT_DIR / "extraction_resultats.csv", index=False)
#
# tiers_df = load_tiers(TIERS_CSV_PATH)
# incoherence_report = build_incoherence_report(extraction_df, tiers_df)
#
# export_final_report(extraction_df, incoherence_report, OUTPUT_DIR)


## 16. Notes, limites & recommandations

- **Revue humaine ciblée** : ne pas automatiser à 100% la recertification. Utiliser
  `nb_incoherences > 0` comme file de travail pour un contrôle manuel, pas comme rejet
  automatique.
- **Seuils ajustables** : `FUZZY_FILENAME_THRESHOLD` (recherche de fichiers) et
  `match_threshold` (comparaison tiers.csv) doivent être calibrés sur un échantillon réel
  avant mise en production.
- **Rotation** : l'approche combine OSD Tesseract (rapide) et fallback VLM (robuste mais plus
  lent) — pour de gros volumes, envisager de ne déclencher le fallback VLM que si l'OSD échoue
  ou a une confiance faible, ce qui est déjà le cas ici.
- **PaddleOCR-VL** n'est pas utilisé ici comme moteur d'extraction final, mais il peut
  avantageusement remplacer/compléter Tesseract pour l'OCR brut et la détection d'orientation,
  notamment sur les documents fortement bilingues arabe/français — à intégrer dans
  `detect_rotation_tesseract` en ajoutant une fonction `detect_rotation_paddleocr_vl` si les
  résultats Tesseract s'avèrent insuffisants sur le corpus réel.
- **Gestion mémoire GPU** : charger les 3 modèles simultanément peut être coûteux ; le cache
  paresseux (`load_model`) + `unload_model` permet de ne garder en mémoire que les modèles
  utilisés récemment si la VRAM est limitée.
- **Traçabilité** : chaque enregistrement conserve `modele_utilise` et `pays_detecte`, utile
  pour l'audit du processus de recertification.
- **Passage à l'échelle** : pour de gros volumes, paralléliser `extract_document` par batch
  d'images (traitement par lot) plutôt qu'image par image, et envisager la quantification
  (int8/4-bit) des modèles si la latence est un enjeu.
